# Kemiripan Gigi Depan (Front View) — Notebook Belajar

Tujuan notebook ini: **memahami mekanika mengukur kemiripan foto gigi depan tanpa label**, sebagai fondasi untuk *grading* IOTN **Aesthetic Component (AC)** lewat pencocokan ke foto referensi.

Kita akan jalan pelan-pelan:

1. Muat & inspeksi foto
2. Preprocessing (kenapa penting)
3. **Toy embedding** (histogram warna) → lihat kenapa ini *gagal*
4. **DINOv2** (encoder semantik) → embedding yang jauh lebih representatif
5. Matriks cosine similarity + nearest-neighbor
6. Bagaimana ini menjadi *AC grader* (anchor, prototype, evaluasi ordinal)
7. Keterbatasan data & langkah berikutnya

> Catatan jujur soal data: folder ini berisi **8 foto acak dari Google**. Itu **cukup untuk belajar pipeline**, tapi **belum cukup untuk membangun atau mengevaluasi grader sungguhan** (butuh 10 foto referensi AC resmi + sedikit data validasi berlabel). Bagian 6–7 menjelaskan ini.

## 0. Ide besar: AC grading = masalah *similarity*, bukan *classification*

Skala IOTN Aesthetic Component punya **10 foto referensi** (grade 1 = paling rapi → grade 10 = paling parah). Cara pakainya: cocokkan foto pasien ke referensi yang **paling mirip** → grade-nya keluar.

Artinya kita **tidak butuh melatih model dari nol**. Kita cukup:

```
encoder pretrained  →  embedding vektor  →  cosine similarity ke 10 anchor  →  1-NN / ranking
```

Semua yang menentukan kualitas ada di **seberapa bagus embedding-nya**. Itu sebabnya pilihan encoder (DINOv2) penting, dan itu yang kita eksplorasi di sini.

In [ ]:
# === Setup ===
# Jalankan di Mac kamu. torch akan otomatis pakai MPS (Apple Silicon) kalau ada.
# Instalasi (sekali saja, di terminal):
#   pip install torch torchvision timm scikit-learn matplotlib pillow

import glob, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

import torch

def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")      # Apple Silicon GPU
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = pick_device()
print("Device:", DEVICE)

# Folder foto (notebook ini diasumsikan berada di folder yang sama dengan foto)
IMG_DIR = "."
paths = sorted(glob.glob(os.path.join(IMG_DIR, "front_google_*.png")))
names = [os.path.basename(p).replace("front_google_","").replace(".png","") for p in paths]
print(f"{len(paths)} foto ditemukan:", names)

## 1. Muat & inspeksi foto

Perhatikan: ukuran beda-beda, ada yang RGBA, sebagian ada **watermark** ("BTeeth", "before"), framing & pencahayaan tidak konsisten. Semua ini nanti "bocor" ke embedding kalau tidak ditangani — poin penting yang akan kita bahas di preprocessing.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, p, n in zip(axes.ravel(), paths, names):
    im = Image.open(p).convert("RGB")
    ax.imshow(im); ax.set_title(f"{n}  {im.size}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Preprocessing

Encoder butuh input seragam. Langkah minimum:

- **RGBA → RGB** (buang alpha)
- **Resize** ke ukuran yang diminta model (DINOv2 ViT-S/14 pakai kelipatan 14; kita pakai 224×224)
- **Normalisasi** mean/std ImageNet

**Yang belum kita lakukan (tapi penting untuk data nyata):**

- **Crop ke ROI gigi** — buang bibir/gusi/background/watermark. Ini biasanya *dampaknya lebih besar* daripada memilih model. Untuk sekarang kita skip (dataset kecil), tapi ingat ini bottleneck utama.
- **Koreksi exposure / white-balance** — foto Google pencahayaannya liar.

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_rgb(path, size=IMG_SIZE):
    im = Image.open(path).convert("RGB").resize((size, size))
    return np.asarray(im, dtype=np.float32) / 255.0   # HxWx3 in [0,1]

def to_model_tensor(arr):
    x = (arr - IMAGENET_MEAN) / IMAGENET_STD           # normalize
    x = torch.from_numpy(x).permute(2, 0, 1)           # HWC -> CHW
    return x.float()

imgs01 = [load_rgb(p) for p in paths]
print("Contoh tensor:", to_model_tensor(imgs01[0]).shape)

## 3. Pemanasan: *toy embedding* — dan kenapa ia gagal

Sebelum model besar, mari bangun embedding paling sederhana: **histogram warna + grid grayscale kasar**. Ini menangkap "warna & terang-gelap", **bukan** struktur gigi (crowding, spacing, protrusion).

Tujuannya bukan supaya bagus — tapi supaya kamu *melihat sendiri* kenapa embedding lemah bikin semua foto tampak mirip.

In [ ]:
def toy_embed(arr):
    g = np.asarray(Image.fromarray((arr*255).astype("uint8")).convert("L").resize((16,16)),
                   dtype=np.float32).flatten() / 255.0
    hist = np.concatenate([np.histogram(arr[:,:,c], bins=16, range=(0,1))[0] for c in range(3)]).astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    v = np.concatenate([g, hist])
    return v / (np.linalg.norm(v) + 1e-8)

toy = np.stack([toy_embed(a) for a in imgs01])
S_toy = cosine_similarity(toy)

plt.figure(figsize=(5.5,4.5))
plt.imshow(S_toy, cmap="viridis", vmin=0, vmax=1)
plt.xticks(range(len(names)), names, rotation=45); plt.yticks(range(len(names)), names)
plt.colorbar(label="cosine similarity"); plt.title("Toy embedding (histogram warna)")
plt.tight_layout(); plt.show()

print("Rentang similarity off-diagonal:",
      f"{S_toy[~np.eye(len(names),dtype=bool)].min():.2f} - {S_toy[~np.eye(len(names),dtype=bool)].max():.2f}")

**Observasi:** semua pasangan ~0.9+ — nyaris tak terbedakan. Embedding ini "buta" terhadap susunan gigi. Inilah alasan kita butuh encoder **semantik** yang belajar fitur bentuk/struktur, bukan sekadar warna. Masuk ke DINOv2.

## 4. DINOv2 — encoder semantik (rekomendasi utama)

**DINOv2 ViT-S/14** (self-supervised, ~21M param) menghasilkan embedding yang kuat untuk kemiripan **tanpa fine-tuning**. Jalan enak di MPS. Kita ambil `CLS` token (vektor global) sebagai embedding per foto.

Load pertama kali akan mengunduh bobot (~85 MB) dari `torch.hub`.

In [ ]:
# Load DINOv2 ViT-S/14 dari torch.hub (butuh internet sekali untuk unduh bobot)
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
model.eval().to(DEVICE)
print("DINOv2 ViT-S/14 loaded. Embedding dim:", model.embed_dim)

In [ ]:
@torch.no_grad()
def dino_embed(arr):
    x = to_model_tensor(arr).unsqueeze(0).to(DEVICE)   # 1x3x224x224
    feat = model(x)                                    # CLS token -> [1, 384]
    v = feat[0].float().cpu().numpy()
    return v / (np.linalg.norm(v) + 1e-8)              # L2-normalize -> cosine ready

dino = np.stack([dino_embed(a) for a in imgs01])
print("Embeddings:", dino.shape)

In [ ]:
S = cosine_similarity(dino)

plt.figure(figsize=(5.5,4.5))
plt.imshow(S, cmap="magma", vmin=S.min(), vmax=1)
plt.xticks(range(len(names)), names, rotation=45); plt.yticks(range(len(names)), names)
plt.colorbar(label="cosine similarity"); plt.title("DINOv2 similarity")
for i in range(len(names)):
    for j in range(len(names)):
        plt.text(j, i, f"{S[i,j]:.2f}", ha="center", va="center",
                 color="white" if S[i,j] < 0.7 else "black", fontsize=7)
plt.tight_layout(); plt.show()

off = S[~np.eye(len(names), dtype=bool)]
print(f"Off-diagonal range: {off.min():.2f} - {off.max():.2f}  (harusnya lebih menyebar daripada toy)")

## 5. Nearest neighbor — "foto paling mirip"

Untuk tiap foto, tampilkan tetangga terdekatnya. Di data nyata, "tetangga" ini adalah **anchor AC** yang menentukan grade.

In [ ]:
fig, axes = plt.subplots(len(names), 2, figsize=(6, 2.2*len(names)))
for i, n in enumerate(names):
    order = np.argsort(-S[i]); nn = [j for j in order if j != i][0]
    axes[i,0].imshow(imgs01[i]); axes[i,0].set_title(f"query {n}", fontsize=9); axes[i,0].axis("off")
    axes[i,1].imshow(imgs01[nn]); axes[i,1].set_title(f"NN {names[nn]}  sim={S[i,nn]:.3f}", fontsize=9); axes[i,1].axis("off")
plt.tight_layout(); plt.show()

## 6. Dari similarity -> **AC grader** (anchor SUDAH tersedia)

10 foto referensi AC sudah dipotong jadi `ac_grade_01.png` ... `ac_grade_10.png` di folder ini
(lihat `crop_references.py`), jadi grader-nya **bisa jalan sungguhan** sekarang.

Ingat tiga penguat penting (tiap grade cuma 1 foto):

1. **Prototype embedding** - kalau nanti punya beberapa foto per grade, rata-ratakan embedding-nya.
2. **Sifat ordinal** - AC skala berurutan (1->10); di bawah kita hitung juga **grade tertimbang** (weighted by similarity), bukan cuma `argmax`.
3. **Evaluasi ordinal** - pakai **MAE** & **within-±1**, butuh set validasi kecil berlabel.

> Catatan: anchor ini masih memuat sedikit overlay angka & belum di-crop ketat ke gigi.
> Untuk hasil lebih bersih, tambahkan ROI crop (lihat Bagian 7).

In [ ]:
import glob

# 10 foto referensi hasil crop_references.py
anchor_paths = sorted(glob.glob("ac_grade_*.png"))
anchor_grades = list(range(1, len(anchor_paths) + 1))
assert len(anchor_paths) == 10, f"harusnya 10 anchor, dapat {len(anchor_paths)}"

anchor_emb = np.stack([dino_embed(load_rgb(p)) for p in anchor_paths])
print("anchor embeddings:", anchor_emb.shape)

@torch.no_grad()
def predict_ac(img_path):
    q = dino_embed(load_rgb(img_path))
    sims = cosine_similarity(q[None], anchor_emb)[0]     # similarity ke tiap grade
    hard = anchor_grades[int(np.argmax(sims))]           # 1-NN
    # grade tertimbang (ordinal): softmax similarity sebagai bobot
    w = np.exp(sims / 0.1); w /= w.sum()
    soft = float(np.dot(w, anchor_grades))
    return hard, soft, sims

print("\nPrediksi AC untuk 8 foto Google:")
for p, n in zip(paths, names):
    hard, soft, sims = predict_ac(p)
    top3 = np.argsort(-sims)[:3]
    print(f"  {n}: AC(1-NN)={hard:>2}  AC(weighted)={soft:4.1f}   "
          f"top-3 grade={[anchor_grades[i] for i in top3]}  sims={np.round(sims[top3],3)}")

# --- Evaluasi (isi kalau sudah punya label) ---
# y_true = [...]; preds = [predict_ac(p)[0] for p in val_paths]
# import numpy as np
# mae = np.mean(np.abs(np.array(y_true) - np.array(preds)))
# within1 = np.mean(np.abs(np.array(y_true) - np.array(preds)) <= 1)
# print("MAE:", round(mae,2), "| within-±1:", round(within1,2))

## 7. Keterbatasan & langkah berikutnya

**Apa yang 8 foto Google ini BISA ajarkan** ✅ mekanika pipeline: encode → similarity → ranking, dan intuisi kenapa encoder semantik penting.

**Apa yang BELUM bisa** ❌
- **Belum ada 10 foto referensi AC resmi** → belum bisa grading sungguhan. *Kebutuhan #1.*
- **Belum ada label validasi** → belum bisa ukur MAE / within-±1. Kumpulkan ~30–50 foto yang di-grade manual (1–2 orang).
- **Belum ada crop ROI gigi** → watermark & bibir mengotori embedding. Tambah deteksi mulut/gigi (mis. mediapipe FaceMesh atau model deteksi gigi) untuk auto-crop.

**Urutan kerja yang disarankan**
1. Kumpulkan 10 foto referensi AC resmi (anchor).
2. Tambahkan crop ROI konsisten.
3. Bikin set validasi kecil berlabel → bandingkan DINOv2 vs baseline (MobileNet) pakai MAE & within-±1.
4. (Opsional) fine-tune triplet/ArcFace kalau sudah ada cukup pasangan berlabel.

## 8. Catatan deployment on-device (untuk nanti)

Prioritas sekarang **akurasi dulu**, jadi DINOv2 di MPS itu tepat. Saat mau ke on-device (iOS/mobile):

- **DINOv2 ViT-S** bisa dikonversi ke **Core ML** (`coremltools`), walau lebih berat dari CNN mobile.
- Alternatif ramah-mobile: **MobileCLIP** (Apple, ada ekspor Core ML resmi) atau **MobileNetV3 / EfficientNet-Lite**.
- Karena arsitekturnya sama (encoder → embedding → cosine ke anchor), kamu bisa **tukar encoder** tanpa mengubah logika grading. Jadi aman memulai dengan DINOv2 sekarang dan mengganti backbone saat optimasi on-device.